# 04 — Train: Auto-ARIMA

Fits one univariate seasonal Auto-ARIMA model for the configured target station and evaluates it once on the test feature artifact.

**Inputs:** train-derived and test-derived feature artifacts  
**Outputs:** in-notebook prediction preview and test metrics only

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports dependencies and fixes the notebook's train-only Auto-ARIMA search bounds, daily seasonal period, artifact paths, preview row count, and feature/target column lists.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from src.config import FORECAST_HORIZON_HOURS, TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

PROCESSED_DIR = Path("data/processed")
PREDICTION_PREVIEW_ROWS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())
TRAIN_WATER_LEVEL_INTERPOLATION = {
    "method": "linear",
    "limit_area": "inside",
    "scope": "training input passed to auto_arima only",
}
AUTO_ARIMA_SEARCH = {
    "seasonal": True,
    "m": 24,
    "information_criterion": "aic",
    "stepwise": True,
    "max_p": 5,
    "max_q": 5,
    "max_P": 2,
    "max_Q": 2,
    "max_d": 2,
    "max_D": 1,
    "max_order": None,
    "out_of_sample_size": 0,
    "error_action": "raise",
    "suppress_warnings": True,
}

## Shared evaluation cohort

Auto-ARIMA uses only water level, but it scores the same strict cohort as Ridge and persistence: complete target vectors marked `target_valid` whose full engineered predictor vectors are present. The predictors therefore define comparability only; they are never passed to the model.

## Helper functions

The helpers validate the scoring cohort, independently verify each chronological physical water-level series, prepare a temporary interpolated training input, calculate metrics, and produce a prediction preview.

In [ ]:
def eligible_rows(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> pd.Series:
    """Return model-ready rows and reject incomplete feature artifacts."""
    required_columns = {"timestamp", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    eligible = frame["target_valid"].eq(True) & frame[FEATURE_COLUMNS].notna().all(
        axis=1
    )
    if frame.loc[eligible, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return eligible

In [ ]:
def water_level_series(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> np.ndarray:
    """Return one UTC-hourly station water-level series with no infinities."""
    required_columns = {"timestamp", "station_id", "water_level"}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )
    if frame.empty:
        raise ValueError(f"{station_id} {artifact_name} artifact is empty")

    timestamp_dtype = frame["timestamp"].dtype
    if (
        not isinstance(timestamp_dtype, pd.DatetimeTZDtype)
        or str(timestamp_dtype.tz) != "UTC"
    ):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be timezone-aware UTC"
        )
    timestamps = pd.DatetimeIndex(frame["timestamp"])
    if timestamps.hasnans:
        raise ValueError(f"{station_id} {artifact_name} timestamps must be complete")
    expected_grid = pd.date_range(timestamps[0], periods=len(timestamps), freq="h")
    if timestamps.has_duplicates or not timestamps.equals(expected_grid):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be unique, ascending, and hourly"
        )

    station_ids = frame["station_id"].drop_duplicates().tolist()
    if station_ids != [station_id]:
        raise ValueError(
            f"{artifact_name} artifact must contain only station {station_id!r}; "
            f"got {station_ids!r}"
        )

    try:
        values = pd.to_numeric(frame["water_level"], errors="raise").to_numpy(
            dtype=float
        )
    except (TypeError, ValueError) as error:
        raise ValueError(
            f"{station_id} {artifact_name} water_level values must be numeric"
        ) from error
    if np.isinf(values).any():
        raise ValueError(
            f"{station_id} {artifact_name} water_level values must not contain infinities"
        )
    return values

In [ ]:
def interpolated_train_water_levels(values: np.ndarray) -> tuple[np.ndarray, int]:
    """Linearly fill internal training gaps for Auto-ARIMA only."""
    missing_rows = int(np.isnan(values).sum())
    interpolated = pd.Series(values).interpolate(
        method=TRAIN_WATER_LEVEL_INTERPOLATION["method"],
        limit_area=TRAIN_WATER_LEVEL_INTERPOLATION["limit_area"],
    )
    prepared = interpolated.to_numpy(dtype=float)
    if not np.isfinite(prepared).all():
        raise ValueError(
            "Auto-ARIMA training input has unfillable water_level gaps at a series boundary"
        )
    return prepared, missing_rows

In [ ]:
def metric_tables(
    actual: pd.DataFrame, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
                "rmse": root_mean_squared_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[target], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(
                    actual[target], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon


def prediction_preview(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    """Return issue timestamps, actual targets, and direct multi-step predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load and validate feature artifacts

Resolve the train/test parquet paths for the target station, failing fast if either is missing. Then validate each physical water-level series independently: it must belong only to the target station, use a contiguous UTC-hourly timeline, and contain numeric values without infinities. Short-gap values marked `imputed=True` and preserved long gaps remain valid physical-artifact values.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
for artifact_path in (train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing feature artifact for {station_id}: {artifact_path}"
        )

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)
train_water_levels = water_level_series(
    train_features, station_id=station_id, artifact_name="train"
)
test_water_levels = water_level_series(
    test_features, station_id=station_id, artifact_name="test"
)

## Apply the eligibility cohort

Restrict metrics to the shared cohort and stop early if either feature split has no usable rows. The Auto-ARIMA fit below uses a temporary interpolation of internal training gaps; it never rewrites the feature artifact.

In [ ]:
train_mask = eligible_rows(train_features, station_id=station_id, artifact_name="train")
test_mask = eligible_rows(test_features, station_id=station_id, artifact_name="test")
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

test_rows = test_features.loc[test_mask]
train_autoarima_values, interpolated_train_rows = interpolated_train_water_levels(
    train_water_levels
)

## Fit the train-only Auto-ARIMA search

This notebook deliberately makes a bounded, train-only AIC order search instead of using a fixed order. Internal gaps are linearly interpolated only in its temporary training input; the method and number of affected rows are displayed below. It uses no validation split, cross-validation, holdout observations, test rows, or engineered predictors for selection. Stepwise search does not enforce a total-order cap.

In [ ]:
autoarima = auto_arima(train_autoarima_values, **AUTO_ARIMA_SEARCH)
search_result = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "train_rows": len(train_water_levels),
            "interpolated_train_rows": interpolated_train_rows,
            "train_interpolation": "linear internal gaps only",
            "order": autoarima.order,
            "seasonal_order": autoarima.seasonal_order,
            "aic": autoarima.aic(),
        }
    ]
)
print(f"Auto-ARIMA train-only AIC search for {station_id}")
display(search_result)
display(pd.DataFrame([TRAIN_WATER_LEVEL_INTERPOLATION]))
display(pd.DataFrame([AUTO_ARIMA_SEARCH]))

## Issue rolling test forecasts

For every chronological test timestamp, first append its observed water level to the fitted SARIMAX state without refitting. A preserved missing value is appended as missing rather than interpolated, so no future test observation is used. Then issue one 24-step forecast. This makes the current observation available whenever it exists while keeping the train-selected parameters fixed.

In [ ]:
state = autoarima.arima_res_
all_test_predictions: list[np.ndarray] = []
for water_level in test_water_levels:
    state = state.append([water_level], refit=False)
    forecast = np.asarray(
        state.get_forecast(steps=FORECAST_HORIZON_HOURS).predicted_mean, dtype=float
    )
    if forecast.shape != (FORECAST_HORIZON_HOURS,):
        raise RuntimeError(
            f"Expected {FORECAST_HORIZON_HOURS} forecast values; got {forecast.shape}"
        )
    all_test_predictions.append(forecast)

test_predictions = np.vstack(all_test_predictions)[test_mask.to_numpy()]

## Evaluate on the test cohort

Build aggregate and per-horizon metric tables, and preview a few predicted rows against actuals.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS], test_predictions, station_id=station_id
)
print(f"Auto-ARIMA test results for {station_id}")
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))